# Prime Guess-r
Pick a mode to start practicing primes from 1 to 100.
Use a gamified interface and batch segmentation to learn more efficiently.

In [1]:
import random, threading, time
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

def sieve(n):
    is_p = [True]*(n+1)
    is_p[0]=is_p[1]=False
    for i in range(2,int(n**0.5)+1):
        if is_p[i]:
            for j in range(i*i,n+1,i):
                is_p[j]=False
    return is_p
IS_PRIME = sieve(100)
def is_prime(n): return IS_PRIME[n]
def smallest_factor(n):
    for i in range(2,n):
        if n%i==0: return i
    return n
def factor_string(n):
    f=smallest_factor(n)
    return f"{f} \u00d7 {n//f}"
def expand_ranges(ranges):
    nums=[]
    for a,b in ranges: nums.extend(range(a,b+1))
    return nums
BATCHES = {"Black":{"ranges":[(1,19)],"color":"#2c2c2c","text":"#ffffff","level":1},
    "Green":{"ranges":[(20,30),(50,60),(80,90)],"color":"#2e7d32","text":"#ffffff","level":2},
    "Red":{"ranges":[(30,40),(60,70)],"color":"#c62828","text":"#ffffff","level":3},
    "Purple":{"ranges":[(40,50),(70,80),(90,100)],"color":"#6a1b9a","text":"#ffffff","level":4},}
for b in BATCHES.values(): b["numbers"]=expand_ranges(b["ranges"])
GLOBAL_COLOR, GLOBAL_TEXT = "#1565c0", "#ffffff"
MENU_COLOR = "#DBEEFF"
TIME_LIMIT = 10
MASTERY_THRESHOLD = 5000
PRIME_COLOR = "#ffd54f"
NOT_PRIME_COLOR = "#8DA3BA"
NEGATIVE_COLOR = "#dc143c"
STREAK_BONUSES = {10:50, 15:75, 20:100, 30:150, 40:250, 50:300, 60:350, 70:400, 80:400, 90:400, 100:500}
BATCH_SCORES = {name:0 for name in BATCHES}
BATCH_MASTERED = {name:False for name in BATCHES}

class GameState:
    def __init__(self):
        self.mode=None
        self.score=0
        self.streak=0
        self.current_number=None
        self.timer_active=False
        self.timer_token=0
        self.score_html=None
        self.streak_html=None
        self.float_html=None
        self.number_html=None
        self.timer_html=None
        self.timer_row=None
        self.buttons_row=None
        self.feedback_html=None
        self.next_row=None
        self.text_color="#ffffff"
state = GameState()
out = widgets.Output()

def get_pool():
    return list(range(1,101)) if state.mode=="global" else BATCHES[state.mode]["numbers"]
def get_bg():
    if state.mode=="global": return GLOBAL_COLOR, GLOBAL_TEXT
    b=BATCHES[state.mode]
    return b["color"], b["text"]
def new_number():
    pool=get_pool()
    n=random.choice(pool)
    tries=0
    while n==state.current_number and len(pool)>1 and tries<10:
        n=random.choice(pool); tries+=1
    state.current_number=n

def number_content(n,color):
    return (f"<div style='text-align:center; margin:35px 0 25px 0;'>"
        f"<span style='font-size:120px; font-weight:700; color:{color}; "
        f"font-family:Poppins, sans-serif; text-shadow:2px 2px 6px rgba(0,0,0,0.3);'>{n}</span></div>")

def timer_content(remaining):
    return (f"<div style='display:inline-block; background:rgba(0,0,0,0.6); color:#ffffff; "
        f"font-weight:800; font-size:24px; font-family:Poppins, sans-serif; padding:6px 20px; "
        f"border-radius:10px; border:2px solid rgba(255,255,255,0.7);'>{remaining}s</div>")

def feedback_content(text,bg_color):
    return (f"<div style='text-align:center; margin-top:10px;'>"
        f"<span style='background:{bg_color}; padding:8px 16px; border-radius:10px; "
        f"font-family:Poppins, sans-serif; font-size:18px; font-weight:600;'>{text}</span></div>")

def streak_content(text_color):
    if state.streak>=2:
        return f"<p style='color:{text_color}; font-size:15px; font-weight:600; margin:2px 0 0 0;'>{state.streak} in a row!</p>"
    return "<p style='margin:2px 0 0 0; height:19px;'></p>"

def float_content(delta,text_color):
    anim_id=f"fl{random.randint(0,999999)}"
    sign="+" if delta>0 else ""
    color=NEGATIVE_COLOR if delta<0 else PRIME_COLOR
    return (f"<style>@keyframes {anim_id} {{0%{{opacity:1; transform:translateY(0);}} "
        f"100%{{opacity:0; transform:translateY(-30px);}}}} .{anim_id} {{animation:{anim_id} 1.1s ease-out forwards;}}</style>"
        f"<div class='{anim_id}' style='text-align:center; font-weight:800; font-size:22px; "
        f"color:{color}; font-family:Poppins, sans-serif;'>{sign}{delta}</div>")

def score_content(text_color):
    if state.mode=="global":
        return f"<p style='color:{text_color}; font-size:18px; margin-top:4px;'>Score: <b>{state.score}</b></p>"
    mastered=BATCH_MASTERED[state.mode]
    badge="<p style='color:#ffd54f; font-weight:700; font-size:16px; margin:2px 0 0 0;'>Batch mastered!</p>" if mastered else ""
    if state.score<0:
        track_color=NEGATIVE_COLOR
        fill=""
        score_color=NEGATIVE_COLOR
    else:
        pct=min(state.score/MASTERY_THRESHOLD*100,100)
        track_color="rgba(255,255,255,0.25)"
        fill=f"<div style='background:#ffd54f;width:{pct}%;height:100%;'></div>"
        score_color=text_color
    return (f"<div style='width:260px;margin:6px auto 0 auto;'>"
        f"<div style='background:{track_color};border-radius:8px;height:14px;overflow:hidden;'>{fill}</div>"
        f"<p style='color:{score_color}; font-size:15px; font-weight:{700 if state.score<0 else 400}; margin:4px 0 0 0;'>{state.score} / {MASTERY_THRESHOLD}</p>{badge}</div>")

def register_score(delta):
    state.score+=delta
    if state.mode!="global":
        BATCH_SCORES[state.mode]=state.score
        if state.score>=MASTERY_THRESHOLD:
            BATCH_MASTERED[state.mode]=True

def start_timer():
    state.timer_token+=1
    state.timer_active=True
    threading.Thread(target=countdown_worker,args=(state.timer_token,),daemon=True).start()

def countdown_worker(token):
    for remaining in range(TIME_LIMIT,0,-1):
        if state.timer_token!=token or not state.timer_active: return
        try: state.timer_html.value=timer_content(remaining)
        except Exception: return
        time.sleep(1)
    if state.timer_token==token and state.timer_active:
        time_up(token)

def time_up(token):
    state.timer_active=False
    state.streak=0
    state.streak_html.value=streak_content(state.text_color)
    state.feedback_html.value=feedback_content("Time out!","#ffe082")
    state.buttons_row.layout.display="none"
    state.timer_row.layout.display="none"
    threading.Thread(target=auto_advance,args=(token,),daemon=True).start()

def auto_advance(token):
    time.sleep(1.5)
    if state.timer_token==token and state.mode is not None:
        new_question()

def new_question():
    new_number()
    state.number_html.value=number_content(state.current_number,state.text_color)
    state.feedback_html.value=""
    state.float_html.value=""
    state.buttons_row.layout.display="flex"
    state.next_row.layout.display="none"
    state.timer_row.layout.display="flex"
    start_timer()

def answer(guess_prime):
    state.timer_active=False
    correct=is_prime(state.current_number)
    color=PRIME_COLOR if correct else NOT_PRIME_COLOR
    state.number_html.value=number_content(state.current_number,color)
    if guess_prime==correct:
        state.streak+=1
        delta=50
        bonus=STREAK_BONUSES.get(state.streak)
        if bonus:
            delta+=bonus
            text=f"Correct! {state.streak} in a row! +{bonus} bonus"
        else:
            text="Correct!"
        register_score(delta)
        state.feedback_html.value=feedback_content(text,"#a5d6a7")
    else:
        state.streak=0
        delta=-100
        register_score(delta)
        if correct:
            text=f"Wrong! {state.current_number} is prime."
        else:
            text=f"Wrong! {state.current_number} = {factor_string(state.current_number)}, not prime."
        state.feedback_html.value=feedback_content(text,"#ef9a9a")
    state.streak_html.value=streak_content(state.text_color)
    state.score_html.value=score_content(state.text_color)
    state.float_html.value=float_content(delta,state.text_color)
    state.buttons_row.layout.display="none"
    state.timer_row.layout.display="none"
    state.next_row.layout.display="flex"

def on_prime_click(b): answer(True)
def on_not_prime_click(b): answer(False)
def on_next_click(b): new_question()
def on_menu_click(b):
    state.timer_active=False
    state.mode=None
    render()
def on_nav_click(mode):
    def handler(b): start_mode(mode)
    return handler

def start_mode(mode):
    state.mode=mode
    state.score=0 if mode=="global" else BATCH_SCORES[mode]
    state.streak=0
    render()
    new_question()

def build_menu():
    title=widgets.HTML("<div style='text-align:center; font-family:Poppins, sans-serif;'>"
        "<h1 style='color:#222;'>Prime Guess-R</h1>"
        "<p style='color:#333; font-size:16px;'>Pick a mode to start practicing primes from 1 to 100</p>"
        "<p style='color:#333; font-size:8px;'> Ⓒ G. Ruquilla</p>"
        "</div>")
    global_btn=widgets.Button(description="Global Mode (1-100)", layout=widgets.Layout(width="280px",height="50px"))
    global_btn.style.button_color=GLOBAL_COLOR
    global_btn.style.text_color="white"
    global_btn.on_click(on_nav_click("global"))
    batch_buttons=[]
    for name,info in BATCHES.items():
        label=f"Level {info['level']} - {name} Batch"
        if BATCH_MASTERED[name]: label+=" (mastered)"
        btn=widgets.Button(description=label, layout=widgets.Layout(width="280px",height="50px"))
        btn.style.button_color=info["color"]
        btn.style.text_color=info["text"]
        btn.on_click(on_nav_click(name))
        batch_buttons.append(btn)
    menu_box=widgets.VBox([title,global_btn]+batch_buttons, layout=widgets.Layout(align_items="center",padding="40px",min_height="480px"))
    menu_box.add_class("prime-menu-box")
    style=widgets.HTML(f"<style>.prime-menu-box {{ background-color: {MENU_COLOR} !important; border-radius: 20px; box-shadow: 0 4px 20px rgba(0,0,0,0.25); }}</style>")
    return widgets.VBox([style,menu_box])

def build_game_shell():
    bg,text_color=get_bg()
    state.text_color=text_color
    mode_label="Global Mode" if state.mode=="global" else f"{state.mode} Batch"
    title_html=widgets.HTML(f"<div style='text-align:center; font-family:Poppins, sans-serif;'>"
        f"<h2 style='color:{text_color}; margin-bottom:0;'>{mode_label}</h2></div>")
    state.score_html=widgets.HTML(score_content(text_color))
    state.streak_html=widgets.HTML(streak_content(text_color))
    state.float_html=widgets.HTML("")
    state.number_html=widgets.HTML(number_content("",text_color))
    state.timer_html=widgets.HTML(timer_content(TIME_LIMIT))
    state.timer_row=widgets.HBox([state.timer_html], layout=widgets.Layout(justify_content="center",margin="0 0 10px 0"))
    prime_btn=widgets.Button(description="Prime", layout=widgets.Layout(width="150px",height="60px"))
    prime_btn.style.button_color="#43a047"
    prime_btn.style.text_color="white"
    prime_btn.on_click(on_prime_click)
    not_prime_btn=widgets.Button(description="Not Prime", layout=widgets.Layout(width="150px",height="60px"))
    not_prime_btn.style.button_color="#e53935"
    not_prime_btn.style.text_color="white"
    not_prime_btn.on_click(on_not_prime_click)
    state.buttons_row=widgets.HBox([prime_btn,not_prime_btn], layout=widgets.Layout(justify_content="center",margin="10px 0"))
    state.feedback_html=widgets.HTML("")
    next_btn=widgets.Button(description="Next", layout=widgets.Layout(width="150px",height="45px"))
    next_btn.style.button_color="#333333"
    next_btn.style.text_color="white"
    next_btn.on_click(on_next_click)
    state.next_row=widgets.HBox([next_btn], layout=widgets.Layout(justify_content="center",margin="10px 0",display="none"))
    menu_btn=widgets.Button(description="Back to Menu", layout=widgets.Layout(width="180px",height="35px"))
    menu_btn.style.button_color="#eeeeee"
    menu_btn.on_click(on_menu_click)
    menu_row=widgets.HBox([menu_btn], layout=widgets.Layout(justify_content="center",margin="20px 0 0 0"))
    game_box=widgets.VBox([title_html,state.score_html,state.streak_html,state.float_html,state.number_html,state.timer_row,
        state.buttons_row,state.feedback_html,state.next_row,menu_row], layout=widgets.Layout(align_items="center",padding="35px",min_height="480px"))
    game_box.add_class("prime-game-box")
    style=widgets.HTML(f"<style>.prime-game-box {{ background-color: {bg} !important; border-radius: 20px; box-shadow: 0 4px 20px rgba(0,0,0,0.25); }}</style>")
    return widgets.VBox([style,game_box])

def render():
    with out:
        clear_output(wait=True)
        if state.mode is None:
            display(build_menu())
        else:
            display(build_game_shell())

display(HTML("<link href='https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700&display=swap' rel='stylesheet'>"))
render()
display(out)

Output()

# Batches Ordering

## <span style="color:#2c2c2c;">Black Batch</span> : The Foundations
<span style="color:#2c2c2c; font-weight:bold;">2  3  5  7  11  13  17  19</span>

No shortcut here, you need to learn these by heart. They're the exceptions to every rule below (2 and 5 are the only primes with an even or 5 last digit), so once they're memorized, everything else follows a pattern.

## <span style="color:#2e7d32;">Green Batch</span> : 3s and 9s
<span style="color:#2e7d32; font-weight:bold;">23 / 29,  53 / 59,  83 / 89</span>

These tens are separated by 30, and their unit digit is always 3 or 9 (3x3). It's all a question of 3s with the Green Batch.

## <span style="color:#c62828;">Red Batch</span> : 1s and 7s
<span style="color:#c62828; font-weight:bold;">31 / 37,  61 / 67</span>

These tens get harder: remember it's the 30s, doubled to the 60s. Unit digit is always 1 or 7.

## <span style="color:#6a1b9a;">Purple Batch</span> : The Oddities
<span style="color:#6a1b9a; font-weight:bold;">41 / 43 / 47,  71 / 73 / 79,  97</span>

These don't follow a single clean digit rule like the other batches." Treat them as a short separate list to memorize on their own.

* Notice the symmetry; the 40s have the 7 (47) and the 70s have the 9 (79).
* Learn 97 on its own. Remember 91 is a trap, it looks prime but is in fact 7 × 13.

### Why this grouping works
Every prime above 5 has to end in 1, 3, 7, or 9. Any other last digit means it's divisible by 2 or 5. That's why the batches line up by "tens position": Green, Red, and most of Purple are the same repeating pattern showing up every 30 numbers (this is sometimes called the "wheel of 30" in number theory).

### Quick check
Black (8) + Green (6) + Red (4) + Purple (7) = 25 primes

---

## FAQ

**What are the batches?**

A segmentation technique to learn primes from 1 to 100 faster and recognise them. Suggested order:
1. Learn the Foundations (Black)
2. Deal with the 3/9 and 1/7 series (Green, Red)
3. Finish with the Oddities (Purple)
4. Switch to Global Mode to practice everything mixed together

**Shouldn't the 90s be with the Red batch?**

No, 91 is an exception (7 × 13), so lumping the 90s into a "ends in 1" rule would teach a false pattern. 97 is safer learned on its own as part of the Oddities.